# Author and deploy an agent with short-term memory using Databricks Lakebase


This notebook demonstrates how to build an agent with short-term memory using the Agent Framework and Lakebase as the agent’s durable memory and checkpoint store. 

Threads allow you to store conversational state in Lakebase so you can pass in thread IDs into your agent instead of needing to send the full conversation history.

In this notebook, you will:
1. Author an agent graph with Lakebase to manage state using thread ids in a Databricks Agent 
2. Wrap the LangGraph agent with `ResponsesAgent` interface to ensure compatibility with Databricks features
3. Test the agent's behavior locally
4. Register model to Unity Catalog, log and deploy the agent for use in Review App, Playground, etc

## Prerequisites
- A Lakebase instance ready and running, see documentation ([AWS](https://docs.databricks.com/aws/en/oltp/create/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/create/)). 
  - You can create a Lakebase instance by going to SQL Warehouses -> Lakebase Postgres -> Create database instance. You will need to retrieve values from the "Connection details" section of your Lakebase to fill out this notebook.
- Complete all the "TODO"s throughout this notebook

### Install dependencies

In [ ]:
%pip install -U -qqqq databricks-langchain[memory] uv databricks-agents mlflow-skinny[databricks]
dbutils.library.restartPython()

## First time setup only: Set up checkpoint tables with your Lakebase instance

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import CheckpointSaver

# --- TODO: Fill in Lakebase project name and branch ---
PROJECT_NAME = "simple-lakebase-centric-workshop"
BRANCH = "production"

with CheckpointSaver(project=PROJECT_NAME, branch=BRANCH) as saver:
    saver.setup()
    print("Checkpoint tables are ready.")

# Define the agent in code

## Write agent code to file agent.py
Define the agent code in a single cell below. This lets you write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.

## Wrap the LangGraph agent using the ResponsesAgent interface
For compatibility with Databricks AI features, the `LangGraphResponsesAgent` class implements the `ResponsesAgent` interface to wrap the LangGraph agent.

Databricks recommends using `ResponsesAgent` as it simplifies authoring multi-turn conversational agents using an open source standard. See MLflow's [ResponsesAgent documentation](https://www.mlflow.org/docs/latest/llms/responses-agent-intro/).

In [ ]:
%%writefile agent.py
import logging
import os
import uuid
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    CheckpointSaver,
)
from databricks.sdk import WorkspaceClient
from langchain_core.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
)

logger = logging.getLogger(__name__)
logging.basicConfig(level=os.getenv("LOG_LEVEL", "INFO"))

LLM_ENDPOINT_NAME = "databricks-claude-opus-4-6"

SYSTEM_PROMPT = """You are an internal support assistant for the data engineering team. You help team members with questions about data pipelines, table schemas, job failures, and platform best practices.

Key guidelines:
- Remember context from earlier in the conversation to avoid asking the user to repeat themselves.
- When discussing tables, always use fully qualified names (catalog.schema.table).
- If a user reports a pipeline failure, ask for the job run ID and error message before troubleshooting.
- Provide concise, actionable answers. Link to relevant documentation when appropriate.
- If you don't know the answer, say so clearly rather than guessing."""

LAKEBASE_PROJECT_NAME = "simple-lakebase-centric-workshop"
LAKEBASE_BRANCH = "production"

tools = []

UC_TOOL_NAMES: list[str] = []
if UC_TOOL_NAMES:
    uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
    tools.extend(uc_toolkit.tools)

VECTOR_SEARCH_TOOLS = []


tools.extend(VECTOR_SEARCH_TOOLS)


class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

class LangGraphResponsesAgent(ResponsesAgent):
    """Stateful agent using ResponsesAgent with pooled Lakebase Autoscaling checkpointing."""

    def __init__(self, project_name: str, branch: str):
        self.workspace_client = WorkspaceClient()
        self.project_name = project_name
        self.branch = branch

        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

    def _create_graph(self, checkpointer: Any):
        def should_continue(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1]
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "continue"
            return "end"

        preprocessor = (
            RunnableLambda(lambda state: [{"role": "system", "content": self.system_prompt}] + state["messages"])
            if self.system_prompt
            else RunnableLambda(lambda state: state["messages"])
        )
        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            return {"messages": [response]}

        workflow = StateGraph(AgentState)
        workflow.add_node("agent", RunnableLambda(call_model))

        if tools:
            workflow.add_node("tools", ToolNode(tools))
            workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
            workflow.add_edge("tools", "agent")
        else:
            workflow.add_edge("agent", END)

        workflow.set_entry_point("agent")
        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        """Get thread_id from request or create a new one.

        Priority:
        1. Use thread_id from custom_inputs if present
        2. Use conversation_id from chat context if available
        3. Generate a new UUID

        Returns:
            thread_id: The thread identifier to use for this conversation
        """
        ci = dict(request.custom_inputs or {})

        if "thread_id" in ci:
            return ci["thread_id"]

        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id

        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        cc_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        langchain_msgs = cc_msgs
        checkpoint_config = {"configurable": {"thread_id": thread_id}}

        with CheckpointSaver(project=self.project_name, branch=self.branch) as checkpointer:
            graph = self._create_graph(checkpointer)

            for event in graph.stream(
                {"messages": langchain_msgs},
                checkpoint_config,
                stream_mode=["updates", "messages"],
            ):
                if event[0] == "updates":
                    for node_data in event[1].values():
                        if len(node_data.get("messages", [])) > 0:
                            yield from output_to_responses_items_stream(node_data["messages"])
                elif event[0] == "messages":
                    try:
                        chunk = event[1][0]
                        if isinstance(chunk, AIMessageChunk) and chunk.content:
                            yield ResponsesAgentStreamEvent(
                                **self.create_text_delta(delta=chunk.content, item_id=chunk.id),
                            )
                    except Exception as exc:
                        logger.error("Error streaming chunk: %s", exc)


# ----- Export model -----
mlflow.langchain.autolog()
AGENT = LangGraphResponsesAgent(LAKEBASE_PROJECT_NAME, LAKEBASE_BRANCH)
mlflow.models.set_model(AGENT)

# Test the Agent locally

In [ ]:
dbutils.library.restartPython()

In [ ]:
from agent import AGENT
result = AGENT.predict({
    "input": [{"role": "user", "content": "I am working on stateful agents"}]
})
print(result.model_dump(exclude_none=True))
thread_id = result.custom_outputs["thread_id"]

In [ ]:
response2 = AGENT.predict({
    "input": [{"role": "user", "content": "What am I working on?"}],
    "custom_inputs": {"thread_id": thread_id}
})
print("Response 2:", response2.model_dump(exclude_none=True))

In [ ]:
response3 = AGENT.predict({
    "input": [{"role": "user", "content": "What am I working on?"}],
})
print("Response 3 No thread id passed:", response3.model_dump(exclude_none=True))

In [ ]:
for chunk in AGENT.predict_stream({
    "input": [{"role": "user", "content": "What am I working on?"}],
    "custom_inputs": {"thread_id": thread_id}
}):
    print("Chunk:", chunk.model_dump(exclude_none=True))

In [ ]:
from agent import AGENT
import mlflow
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ChatContext
)

conversation_id = "e12345-b237-484f-ad6e-f000551703f5"

req = ResponsesAgentRequest(
    input=[{"role": "user", "content": "I am working on stateful agents, and I love the beaches, specially white sand beaches. I don't like mountains"}],
    context=ChatContext(
        conversation_id=conversation_id,
        user_id="ananyaroy@databricks.com"
    )
)
result = AGENT.predict(req)

print(result.model_dump(exclude_none=True))
thread_id = result.custom_outputs["thread_id"]
print(f"Resolved thread_id from agent: {thread_id}")

In [ ]:
for chunk in AGENT.predict_stream({
    "input": [{"role": "user", "content": "What am I working on , and is there any personal preference i have shared with you earlier?"}],
    "custom_inputs": {"thread_id": conversation_id}
}):
    print("Chunk:", chunk.model_dump(exclude_none=True))

# Log the agent as an MLflow model
Log the agent as code from the agent.py file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

## Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model()`.

**TODO:** 
- Add lakebase as a resource type
- If your Unity Catalog tool queries a [vector search index](https://docs.databricks.com/docs%20link) or leverages [external functions](https://docs.databricks.com/docs%20link), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)).

In [ ]:
import mlflow
from agent import tools, LLM_ENDPOINT_NAME, LAKEBASE_PROJECT_NAME, LAKEBASE_BRANCH
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint, DatabricksLakebase
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool
from pkg_resources import get_distribution

resources = [
    DatabricksServingEndpoint(LLM_ENDPOINT_NAME),
    DatabricksLakebase(database_instance_name=LAKEBASE_PROJECT_NAME),
]

for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "input": [
        {
            "role": "user",
            "content": "What is an LLM agent?"
        }
    ],
    "custom_inputs": {"thread_id": "test-thread"},
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            f"databricks-langchain[memory]=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ]
    )

# Evaluate the agent with Agent Evaluation
Use Agent Evaluation to evalaute the agent's responses based on expected responses and other evaluation criteria. Use the evaluation criteria you specify to guide iterations, using MLflow to track the computed quality metrics. See Databricks documentation ([AWS](https://docs.databricks.com/(https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)).

To evaluate your tool calls, add custom metrics. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)).

In [ ]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Calculate the 15th Fibonacci number"}]},
        "expected_response": "The 15th Fibonacci number is 610.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],
)


# Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the mlflow.models.predict() API.

In [ ]:
import json
import mlflow

loaded_model = mlflow.pyfunc.load_model(f"runs:/{logged_agent_info.run_id}/agent")

test_thread_id = "pre-deploy-validation-thread-v2"

print("=== Turn 1: Establish context ===")
result1 = loaded_model.predict(
    {
        "input": [{"role": "user", "content": "My name is X and I'm debugging a pipeline failure in engen_prod.silver_member.member_dim"}],
        "custom_inputs": {"thread_id": test_thread_id},
    }
)
print(json.dumps(result1, indent=2, default=str))

print("\n=== Turn 2: Test memory (same thread) ===")
result2 = loaded_model.predict(
    {
        "input": [{"role": "user", "content": "What table was I asking about, and what's my name?"}],
        "custom_inputs": {"thread_id": test_thread_id},
    }
)
print(json.dumps(result2, indent=2, default=str))

print("\n=== Turn 3: No memory (different thread) ===")
result3 = loaded_model.predict(
    {
        "input": [{"role": "user", "content": "What table was I asking about, and what's my name?"}],
        "custom_inputs": {"thread_id": "different-thread-no-context"},
    }
)
print(json.dumps(result3, indent=2, default=str))

# Register the model to Unity Catalog
Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [ ]:
mlflow.set_registry_uri("databricks-uc")

catalog = "catalog"
schema = "schema"
model_name = "short-term-memory-agent"

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

Deploy the agent

In [ ]:
from databricks import agents
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags = {"endpointSource": "docs"}, deploy_feedback_model=False)

# Next steps
It will take around 15 minutes for you to finish deploying your agent. After your agent is deployed, you can chat with it in Review App/playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. 

Query your Lakebase instance to see a record of your conversation at various threads/checkpoints. Here is a basic query to see the 10 most recent checkpoints:

```
SELECT
    c.*,
    (c.checkpoint::json->>'ts')::timestamptz AS ts
FROM checkpoints c
ORDER BY ts DESC
LIMIT 10;
```